# Blues Lick Finder: RAG Baseline

A minimal RAG baseline over blues guitar lesson videos, following the same structure we used in the module: a `search` function, a `build_prompt` function, an `llm` function, and a top-level `rag` function that chains them together.

Given a song or technique, it retrieves candidate lessons and asks the LLM to rank them by teaching quality.

In [1]:
import json

from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from minsearch import Index
from gitsource import chunk_documents

openai_client = OpenAI()

## 1. Load Your Data

Each document is one YouTube lesson video: metadata from a `yt-dlp` search (no API key needed) plus the English transcript from `youtube-transcript-api` when one exists. Each video also gets its top 20 viewer comments by likes, excluding the creator's own.

The first run fetches from YouTube and caches everything to `data/lessons.json`. Later runs load from that file. Delete it to re-fetch.

The fetching code lives in `youtube_data.py` next to this notebook.

In [2]:
from youtube_data import load_lessons

documents = load_lessons()

with_transcript = sum(1 for d in documents if d["transcript"])
with_comments = sum(1 for d in documents if d["comments"])
print(f"Loaded {len(documents)} lessons, {with_transcript} with transcripts, {with_comments} with comments")

Loaded 40 lessons, 27 with transcripts, 39 with comments


## 2. Build the Index

Transcripts are long, so we split them into overlapping chunks with `chunk_documents` (3000 characters, step 1500), the same way we did in class, and index the chunks instead of whole videos. Each chunk keeps its video's metadata. Videos without a transcript are skipped.

minsearch builds an in-memory TF-IDF index over the fields you specify.

In [3]:
transcript_docs = [doc for doc in documents if doc["transcript"]]
document_chunks = chunk_documents(
    transcript_docs, size=3000, step=1500, content_field_name="transcript"
)
print(f"{len(transcript_docs)} videos -> {len(document_chunks)} chunks")

index = Index(
    text_fields=["title", "description", "tags", "content"],
    keyword_fields=["video_id", "channel", "topic"],
)

index.fit(document_chunks)

27 videos -> 142 chunks


## 3. Define the RAG Functions

`search` retrieves relevant documents, `build_prompt` formats them into a prompt, `llm` calls the model, and `rag` chains all three.

In [4]:
def search(query):
    results = index.search(query, num_results=50)

    # Results are sorted best-first and may hold several chunks of the same
    # video. Keep only the first (best) chunk per video so we return 5
    # different videos.
    best_chunk_per_video = {}
    for chunk in results:
        best_chunk_per_video.setdefault(chunk["video_id"], chunk)

    return list(best_chunk_per_video.values())[:5]

In [ ]:
instructions = """
You're a blues guitar teacher helping a student find the best lesson videos
for the song or technique in their QUERY. YouTube ranks videos by engagement;
your job is to rank them by teaching quality instead.

Rank exactly the candidates in the CONTEXT. Never add, repeat, or rename a
lesson that isn't one of the candidates.

Use only the candidate lessons in the CONTEXT. Skip candidates that aren't
about the requested song or technique. Rank the rest from best to worst
using this rubric, judged mostly from each candidate's `content` field,
which is the part of the video's transcript that best matches the QUERY:

- Breakdown: splits the lick into small, clearly explained pieces
- Demonstration: plays it slowly before playing it at full speed
- Specifics: names frets, strings, fingers, or points to tab/notation
- Why: explains the theory or feel (scale, chord tones, phrasing) behind it
- Focus: the excerpt teaches the lick instead of intros or sales pitches

Each candidate also has a `comments` field with its most-liked viewer
comments and like counts. Use comments that talk about the teaching (pace,
clarity, tabs, how quickly it gets going, mistakes) as extra evidence for
the rubric. Ignore general praise, jokes, and song requests. An empty list
means no comments are available. Missing or empty comments are neutral:
never lower a score just because a video has no comments.

When you use a comment, quote it word for word with its like count. Never
describe or summarize what viewers said without quoting them, and never
claim something about the comments that no comment actually says.

Also estimate the lesson's level from the title, tags, transcript excerpt,
and comments:

- beginner: explains basics like what a bend or the pentatonic box is,
  plays very slowly, or says it's for people starting out
- intermediate: assumes the pentatonic box and basic bends are known and
  focuses on phrasing, licks in context, and following the chord changes
- advanced: chord-tone soloing, fast or complex runs, little explanation
  of the basics

Give one short reason for the level, quoting the transcript or a comment
when you can. If there isn't enough to tell, say "unclear" instead of
guessing.

For each ranked lesson, return: rank, title, channel, URL, a score from 1 to 5,
a one-sentence justification that cites the rubric, viewer evidence (1-2
quoted comments with like counts, or "none" if no comment is about the
teaching), level with a short reason, and practice-log tags. Tags are real
values, not field names, e.g. "Tags: BB King box, bending, key of A".
Include a key only if a candidate names one. If none of the candidates fit
the QUERY, say so.
""".strip()


def build_prompt(query, search_results):
    search_result_json = json.dumps(search_results, indent=2)

    user_prompt = f"""
<QUERY>
{query}
</QUERY>

<CONTEXT>
{search_result_json}
</CONTEXT>
""".strip()

    return user_prompt

In [6]:
def llm(user_prompt, instructions=None, model='gpt-4o-mini'):
    messages = []

    if instructions is not None:
        messages.append({
            "role": "system",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    response = openai_client.responses.create(
        model=model,
        input=messages
    )

    return response.output_text

In [7]:
def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(prompt, instructions)
    return answer

## 4. Try It Out

In [8]:
print(rag("Which lessons teach the BB King box licks best?"))

Here are the best-ranked lessons teaching BB King box licks based on the provided candidates:

1. **Title**: [Learn 10 Classic Blues Licks in the BB King Box!](https://www.youtube.com/watch?v=i6Ho6aubraw)  
   **Channel**: swiftlessons  
   **Score**: 5  
   **Justification**: This lesson provides a thorough breakdown of 10 licks, clearly explaining each lick by splitting it into easy-to-understand parts with specifics on finger placement and fret numbers. It also emphasizes the practical application of these licks in a blues context.  
   **Viewer Evidence**: "one of the best teachers on the net, very usable and slow enough to learn, not just a fast short tease" (16 likes).  
   **Level**: Intermediate, as it builds on the concept of the BB King box and assumes some prior knowledge of basic licks.  
   **Tags**: technique, BB King box, song (blues progression in G).

2. **Title**: [The BB King Box & Beyond - Blues Lead Guitar Lesson](https://www.youtube.com/watch?v=pZz-QHt8UBc)  
   *